# Databases
- connecting
- creating
- selecting
- updating
- scraping

In [ ]:
! python -m pip install mysql-connector-python

In [ ]:
import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()

#le me do le stuff

cursor.close()
dbconnection.close()


## creating data in the DB

In [1]:
#ADDING 1 NAME TO DB

import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()

query = "INSERT INTO actors(name) VALUES('Vin Diesel');"
cursor.execute(query)
dbconnection.commit()
#VERY IMPORTANT THAT YOU COMMIT THIS STUFF!

cursor.close()
dbconnection.close()

In [2]:
#ADDING MULTIPLE NAMES TO DB

import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()

query = "INSERT INTO actors(name) VALUES(%s);"
data = [["Jacky Chan"], ["Robert De Niro"], ["Benedict Cumberbatch"], ["Tom Hardy"]]

cursor.executemany(query, data)
dbconnection.commit()
#VERY IMPORTANT THAT YOU COMMIT THIS STUFF!

cursor.close()
dbconnection.close()

In [3]:
#ADDING MOVIES AND DESCRIPTIONS TP DB

import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()

query = "INSERT INTO movies(title, synopsis) VALUES(%s, %s);"
data = ("FastAndTheFurious2", "People racing cars")
cursor.execute(query, data)

movies = [("Karate Kid", "KARATE"), ("Good fellas", "Italian gangster"), ("Sherlock Holmes", "Solving mystery crimes"),
("Legend", "Twin brothers that are raising a gangster group in London in 1960")]

for movie in movies:
    cursor.execute(query, movie)

dbconnection.commit()
#VERY IMPORTANT THAT YOU COMMIT THIS STUFF!

cursor.close()
dbconnection.close()

## Reading operations

In [ ]:
import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)
cursor = dbconnection.cursor()
query = "SELECT * from actors;"
cursor.execute(query)

data = cursor.fetchall()
print(data)

for actor in data:
    print(actor)

for id, actor in data:
    print(f"Possible actor: {actor}")

for actor in data:
    print(actor[1])

cursor.close()
dbconnection.close()

[(1, 'Michelle Yeoh'), (2, 'Stephan James'), (3, 'Jamie Lee Curtis'), (4, 'Tom Cruise'), (5, 'Vin Diesel'), (6, 'Jacky Chan'), (7, 'Robert De Niro'), (8, 'Benedict Cumberbatch'), (9, 'Tom Hardy')]
(1, 'Michelle Yeoh')
(2, 'Stephan James')
(3, 'Jamie Lee Curtis')
(4, 'Tom Cruise')
(5, 'Vin Diesel')
(6, 'Jacky Chan')
(7, 'Robert De Niro')
(8, 'Benedict Cumberbatch')
(9, 'Tom Hardy')
Possible actor: Michelle Yeoh
Possible actor: Stephan James
Possible actor: Jamie Lee Curtis
Possible actor: Tom Cruise
Possible actor: Vin Diesel
Possible actor: Jacky Chan
Possible actor: Robert De Niro
Possible actor: Benedict Cumberbatch
Possible actor: Tom Hardy
Michelle Yeoh
Stephan James
Jamie Lee Curtis
Tom Cruise
Vin Diesel
Jacky Chan
Robert De Niro
Benedict Cumberbatch
Tom Hardy


In [ ]:
import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)
cursor = dbconnection.cursor()
query = """SELECT movies.title, actors.name from movies
 inner join movie_actor on movies.id = movie_actor.movie_id
 inner join actors on actors.id = movie_actor.actor_id;
"""

cursor.execute(query)
data = cursor.fetchall()

for movie, actor in data:
    print(f"{actor} played a fantastic role in {movie}!")

cursor.close()
dbconnection.close()

Michelle Yeoh played a fantastic role in Everything Everywhere All at Once!
Stephan James played a fantastic role in Everything Everywhere All at Once!
Jamie Lee Curtis played a fantastic role in Everything Everywhere All at Once!
Tom Cruise played a fantastic role in Top Gun: Maverick!


## UPDATE operations

In [12]:
import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)
cursor = dbconnection.cursor()
#cursor.execute("INSERT INTO movies(title) values('The Wolf of Wall Street');")
cursor.execute("""UPDATE movies SET synopsis = 'Poor guy learning about the big bucks and finally 
               abusing his power and money.' WHERE title like 'The Wolf of Wall Street';""")
dbconnection.commit()

cursor.close()
dbconnection.close()

### Scraping data from the oscars into our database

In [21]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_Academy_Award-winning_films"
headers = {"User-Agent" : "Anthony"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text)

import mysql.connector
dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)
cursor = dbconnection.cursor()

tables = soup.find_all("table")
rows = tables[0].find_all("tr")
for row in rows:
    tds = row.find_all("td")
    if len(tds) > 0:
        title = tds[0].get_text().strip().replace("'", "")
        year = tds[1].get_text().strip()
        awards = tds[2].get_text().strip()
        nom = tds[3].get_text().strip()

        #first just to be sure: check if the movie is already in the database
        query = f"SELECT * from movies WHERE title like '{title}'"
        cursor.execute(query)
        data = cursor.fetchall()
        
        if len(data) == 0:
            query = f"INSERT into movies(title) values ('{title}')"
            cursor.execute(query)
            dbconnection.commit()

            movie_id = cursor.lastrowid
            query = f"""INSERT into awards(awardshow_id, year, movie_id, awards, nominations)
             values(%s, %s, %s, %s, %s)"""
            values = (1, year, movie_id, awards, nom)
            cursor.execute(query, values)
            dbconnection.commit()

cursor.close()
dbconnection.close()